In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSE720/code')
import os, glob
import pandas as pd
import numpy as np
from scipy import stats
from config import Config
cfg = Config()

Mounted at /content/drive


In [ ]:
csvs = glob.glob(os.path.join(cfg.eval_dir, '*_per_item_results.csv'))
print("Found per-item result files:")
for c in csvs:
    print(" -", os.path.basename(c))
assert any('EyeGAN' in c for c in csvs), "Run 01_FullTestSet_Evaluation.ipynb for EyeGAN first."

dfs = {os.path.basename(c).replace('_per_item_results.csv', ''): pd.read_csv(c) for c in csvs}

Found per-item result files:
 - EyeGAN_per_item_results.csv


In [ ]:
def cohens_d(a, b):
    n1, n2 = len(a), len(b)
    pooled_std = np.sqrt(((n1-1)*np.var(a, ddof=1) + (n2-1)*np.var(b, ddof=1)) / (n1+n2-2))
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std > 0 else float('nan')

def compare(eyegan_vals, baseline_vals, metric_name, baseline_name):
    t_stat, t_p = stats.ttest_ind(eyegan_vals, baseline_vals, equal_var=False)
    u_stat, u_p = stats.mannwhitneyu(eyegan_vals, baseline_vals, alternative='two-sided')
    d = cohens_d(eyegan_vals, baseline_vals)
    return {
        'metric': metric_name, 'baseline': baseline_name,
        'eyegan_mean': np.mean(eyegan_vals), 'baseline_mean': np.mean(baseline_vals),
        'welch_t': t_stat, 'welch_p': t_p,
        'mannwhitney_u': u_stat, 'mannwhitney_p': u_p,
        'cohens_d': d,
        'significant_at_0.05': bool(t_p < 0.05 and u_p < 0.05),
    }

results = []
eyegan_df = dfs['EyeGAN']
for baseline_name, bdf in dfs.items():
    if baseline_name == 'EyeGAN':
        continue
    for metric in ['psnr', 'ssim', 'mse']:
        if metric not in bdf.columns:
            continue
        results.append(compare(eyegan_df[metric].dropna().values, bdf[metric].dropna().values,
                                metric, baseline_name))

stats_df = pd.DataFrame(results)
pd.set_option('display.width', 140)
print(stats_df.round(4).to_string(index=False))

out_path = os.path.join(cfg.eval_dir, 'statistical_significance_results.csv')
stats_df.to_csv(out_path, index=False)
print(f"\nSaved to {out_path}")

Empty DataFrame
Columns: []
Index: []

Saved to /content/drive/MyDrive/CSE720/revision/evaluation/statistical_significance_results.csv


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
from config import Config

cfg = Config()
out_dir = cfg.eval_dir
ablation_dir = cfg.ablation_dir
out_path = os.path.join(out_dir, 'statistical_significance_results.csv')

# 1. Load ground truth EyeGAN results
eyegan_path = os.path.join(out_dir, 'EyeGAN_per_item_results.csv')

if not os.path.exists(eyegan_path):
    print(f"Error: Could not find '{eyegan_path}'.")
    print("Please run Notebook 01 first to generate the full test set evaluation CSV.")
else:
    eyegan_df = pd.read_csv(eyegan_path)

    # 2. Collect baseline and ablation CSV files from both directories
    eval_files = [os.path.join(out_dir, f) for f in os.listdir(out_dir) if f.endswith('_per_item_results.csv') and f != 'EyeGAN_per_item_results.csv']

    ablation_files = []
    if os.path.exists(ablation_dir):
        ablation_files = [os.path.join(ablation_dir, f) for f in os.listdir(ablation_dir) if f.endswith('_per_item_results.csv')]

    all_comparison_files = eval_files + ablation_files

    if len(all_comparison_files) == 0:
        print("Error: No baseline or ablation per-item CSV files found.")
        print("Please ensure Notebook 02 and Notebook 03 (Ablations) have completed successfully.")
    else:
        stats_rows = []
        metrics = ['psnr', 'ssim', 'mse']

        for file_path in all_comparison_files:
            model_name = os.path.basename(file_path).replace('_per_item_results.csv', '')
            c_df = pd.read_csv(file_path)

            # Merge on matching image and target domain
            merged = pd.merge(
                eyegan_df, c_df,
                on=['image', 'source', 'target'],
                suffixes=('_eyegan', '_model')
            )

            for m in metrics:
                eyegan_vals = merged[f'{m}_eyegan'].dropna().values
                model_vals = merged[f'{m}_model'].dropna().values

                if len(eyegan_vals) > 0 and len(model_vals) > 0:
                    # Welch's t-test (unequal variances)
                    t_stat, p_val = stats.ttest_ind(eyegan_vals, model_vals, equal_var=False)

                    # Cohen's d effect size
                    n1, n2 = len(eyegan_vals), len(model_vals)
                    s1, s2 = np.std(eyegan_vals, ddof=1), np.std(model_vals, ddof=1)
                    s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
                    cohens_d = (np.mean(eyegan_vals) - np.mean(model_vals)) / s_pooled if s_pooled > 0 else 0.0

                    stats_rows.append({
                        'metric': m.upper(),
                        'baseline': model_name,
                        'eyegan_mean': np.mean(eyegan_vals),
                        'model_mean': np.mean(model_vals),
                        'welch_t': t_stat,
                        'welch_p': p_val,
                        'cohens_d': cohens_d
                    })

        # 3. Create DataFrame and apply FDR Correction (Benjamini-Hochberg)
        stats_df = pd.DataFrame(stats_rows)

        if not stats_df.empty and 'welch_p' in stats_df.columns:
            p_vals = stats_df['welch_p'].fillna(1.0).values
            reject, pvals_corrected, _, _ = multipletests(p_vals, method='fdr_bh', alpha=0.05)

            stats_df['welch_p_fdr_corrected'] = pvals_corrected
            stats_df['significant_after_fdr'] = reject

            cols_to_show = ['metric', 'baseline', 'welch_p', 'welch_p_fdr_corrected', 'significant_after_fdr', 'cohens_d']

            print("\n--- Statistical Significance Test Results (FDR Corrected) ---")
            print(stats_df[cols_to_show].round(4).to_string(index=False))

            stats_df.to_csv(out_path, index=False)
            print(f"\nSuccessfully saved statistical results to: {out_path}")


--- Statistical Significance Test Results (FDR Corrected) ---
metric             baseline  welch_p  welch_p_fdr_corrected  significant_after_fdr  cohens_d
  PSNR      ablation_no_cls   0.0000                 0.0000                   True   -2.8480
  SSIM      ablation_no_cls   0.0000                 0.0000                   True   -2.8073
   MSE      ablation_no_cls   0.0000                 0.0000                   True    0.4134
  PSNR    ablation_no_cycle   0.6869                 0.6869                  False   -0.0180
  SSIM    ablation_no_cycle   0.0001                 0.0002                   True   -0.1700
   MSE    ablation_no_cycle   0.0147                 0.0161                   True    0.1092
  PSNR ablation_no_identity   0.0000                 0.0000                   True    1.5907
  SSIM ablation_no_identity   0.0000                 0.0000                   True    1.6283
   MSE ablation_no_identity   0.0000                 0.0000                   True   -0.8611
  PSNR 